# 01 — EDA (Home Credit Default Risk)

**Objetivo:** caracterizar en profundidad `application_*`: `TARGET` y baselines ingenuos, tipos y cardinalidad, nulos globales y **nulos por clase `TARGET`**, alineación train/test, **deriva (KS)** en numéricas, IDs, volumetría de tablas auxiliares, sanity checks y percentiles, correlaciones y **heatmap**, categóricas (**Cramér V**), tasas de default por grupo, **EXT_SOURCE** por clase, y **bureau** (filas por cliente vs default). Modelado en notebooks posteriores.

**Problema:** predecir incumplimiento (`TARGET`=1). Métrica Kaggle: **ROC-AUC**.


## Fuentes de datos

| Recurso | Uso |
|--------|-----|
| `data/raw/*.csv` | CSV del competición |
| `configs/home_credit.yaml` | Archivos, claves, `TARGET` |
| `configs/base.yaml` | Semilla, splits |
| `HomeCredit_columns_description.csv` | Diccionario de columnas |

Código: `src.data.home_credit`.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy.stats import chi2_contingency, ks_2samp

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists() and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display

from src.data.home_credit import keys, read_table, table_path, target_column
from src.data.quality import basic_range_checks, missing_summary
from src.utils.paths import project_root
from src.utils.seed import set_seed

plt.style.use("ggplot")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)

set_seed(42)
print("project_root:", project_root())


In [ ]:
train = read_table("application_train")
test = read_table("application_test")

y_col = target_column()
k_curr = keys()["sk_id_curr"]

print("train:", train.shape, "| test:", test.shape)
print("TARGET en train:", y_col in train.columns)
print("TARGET en test:", y_col in test.columns)


## Tabla principal `application_train` / `application_test`

- Una fila por solicitud (`SK_ID_CURR`).
- `TARGET` solo en train.
- Numéricas, categóricas y flags.


In [ ]:
vc = train[y_col].value_counts().sort_index()
n0, n1 = int(vc.get(0, 0)), int(vc.get(1, 0))
imbalance = n1 / len(train) if len(train) else 0
ratio_neg_pos = n0 / n1 if n1 else np.nan

print("Distribución TARGET:", vc.to_string())
print(f"Proporción positivos (TARGET=1): {imbalance:.4f}")
print(f"Ratio clases 0:1 (aprox.): {ratio_neg_pos:.2f}:1")
print("Baseline siempre 0: accuracy ~", f"{1 - imbalance:.4f}", "(mayoría); P/R clase 1 = 0")
print("Baseline siempre 1: accuracy ~", f"{imbalance:.4f}", "| recall 1 = 1 | precision ~", f"{imbalance:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
vc.plot(kind="bar", ax=axes[0], color=["#4c78a8", "#f58518"])
axes[0].set_xlabel("TARGET")
axes[0].set_ylabel("Recuento")
axes[0].set_title("Conteos (train)")
axes[1].pie(
    [n0, n1],
    labels=["TARGET=0", "TARGET=1"],
    autopct="%1.1f%%",
    colors=["#4c78a8", "#f58518"],
    startangle=90,
)
axes[1].set_title("Proporción")
plt.tight_layout()


### Tipos y columnas

Dtypes, cardinalidad `object`, flags binarios, columnas casi constantes.


In [ ]:
dtypes_train = train.dtypes.value_counts()
print("Dtypes en train:")
print(dtypes_train)

feat_cols = [c for c in train.columns if c != y_col]
num_feats = train[feat_cols].select_dtypes(include=[np.number]).columns.tolist()
obj_cols = train[feat_cols].select_dtypes(include=["object"]).columns.tolist()

print("Numéricas:", len(num_feats), "| Object:", len(obj_cols))

binary_like = []
for c in num_feats:
    u = train[c].dropna().unique()
    if len(u) <= 2:
        ufloat = {float(x) for x in u}
        if ufloat <= {0.0, 1.0}:
            binary_like.append(c)
print("Columnas numéricas binarias (solo 0/1):", len(binary_like))

card = train[obj_cols].nunique(dropna=False).sort_values(ascending=False)
print("Top 15 por cardinalidad (object):")
print(card.head(15))

nunique_all = train[feat_cols].nunique(dropna=False)
constant_cols = nunique_all[nunique_all <= 1].index.tolist()
print("Columnas con <=1 valor distinto:", len(constant_cols), constant_cols[:15])


## Valores faltantes

Fracción de NA global y comparación **TARGET=0 vs TARGET=1** en las columnas con más nulos.


In [ ]:
miss = missing_summary(train).sort_values(ascending=False)
miss_nonzero = miss[miss > 0]
print(f"Columnas con al menos un nulo: {len(miss_nonzero)} / {len(train.columns)}")
display(miss_nonzero.head(25).to_frame("frac_na"))

fig, ax = plt.subplots(figsize=(8, 5))
miss_nonzero.head(25).plot(kind="barh", ax=ax, legend=False, color="#4c78a8")
ax.set_xlabel("Fracción NA")
ax.set_title("Top 25 columnas por % de nulos (train)")
plt.tight_layout()

cols_miss = miss_nonzero.head(30).index.tolist()
rows = []
for col in cols_miss:
    if col == y_col:
        continue
    m0 = train.loc[train[y_col] == 0, col].isna().mean()
    m1 = train.loc[train[y_col] == 1, col].isna().mean()
    rows.append({"column": col, "na_rate_t0": m0, "na_rate_t1": m1, "diff": abs(m1 - m0)})
miss_target = pd.DataFrame(rows).sort_values("diff", ascending=False)
print("Top 15 columnas donde la tasa de NA difiere más entre TARGET 0 y 1:")
display(miss_target.head(15))


## Calidad: duplicados e IDs

`SK_ID_CURR` único en train y en test; sin solapamiento train∩test.


In [ ]:
dup_train = train[k_curr].duplicated().sum()
dup_test = test[k_curr].duplicated().sum()
overlap = set(train[k_curr]) & set(test[k_curr])

print(f"Filas duplicadas por {k_curr} en train: {dup_train}")
print(f"Filas duplicadas por {k_curr} en test: {dup_test}")
print(f"IDs compartidos train∩test: {len(overlap)}")


### Alineación train / test

Columnas exclusivas de cada tabla y dtypes en el cruce común (sin `TARGET`).


In [ ]:
only_train = set(train.columns) - set(test.columns)
only_test = set(test.columns) - set(train.columns)
common = (set(train.columns) & set(test.columns)) - {y_col}

print("Solo en train:", only_train)
print("Solo en test:", only_test)
print(f"Columnas comunes (sin TARGET): {len(common)}")

dtype_mismatch = []
for col in sorted(common):
    if train[col].dtype != test[col].dtype:
        dtype_mismatch.append((col, train[col].dtype, test[col].dtype))
print(f"Diferencias de dtype en comunes: {len(dtype_mismatch)}")
if dtype_mismatch:
    display(pd.DataFrame(dtype_mismatch, columns=["column", "dtype_train", "dtype_test"]).head(25))


### Coherencia train vs test (exploratorio)

Medias en numéricas comunes y test **Kolmogorov-Smirnov** (muestra hasta 50k filas por serie). *p* bajo sugiere distribuciones distintas.


In [ ]:
num_common = [
    col
    for col in common
    if col in train.columns
    and pd.api.types.is_numeric_dtype(train[col])
    and pd.api.types.is_numeric_dtype(test[col])
]
rows = []
for col in num_common:
    t_mean = train[col].mean()
    s_mean = test[col].mean()
    diff_rel = abs(t_mean - s_mean) / (abs(t_mean) + 1e-12)
    sub_t = train[col].dropna()
    sub_s = test[col].dropna()
    if len(sub_t) > 50000:
        sub_t = sub_t.sample(50000, random_state=42)
    if len(sub_s) > 50000:
        sub_s = sub_s.sample(50000, random_state=42)
    if len(sub_t) < 2 or len(sub_s) < 2:
        continue
    ks_stat, ks_p = ks_2samp(sub_t, sub_s)
    rows.append(
        {
            "column": col,
            "mean_train": t_mean,
            "mean_test": s_mean,
            "rel_mean_diff": diff_rel,
            "ks_statistic": ks_stat,
            "ks_pvalue": ks_p,
        }
    )
drift = pd.DataFrame(rows).sort_values("ks_statistic", ascending=False)
print("Top 15 por estadístico KS:")
display(drift.head(15))
print("Top 10 por divergencia relativa de medias:")
display(drift.sort_values("rel_mean_diff", ascending=False).head(10)[["column", "mean_train", "mean_test", "rel_mean_diff"]])


## Otras tablas (volumen)

Filas por CSV auxiliar (comportamiento = múltiples filas por cliente).


In [ ]:
def count_csv_rows(path: Path) -> int:
    with path.open("rb") as f:
        return sum(1 for _ in f) - 1


aux_keys = [
    "bureau",
    "bureau_balance",
    "previous_application",
    "pos_cash_balance",
    "installments_payments",
    "credit_card_balance",
]
rows = {}
for k in aux_keys:
    rows[k] = count_csv_rows(table_path(k))

pd.Series(rows, name="n_rows").sort_values(ascending=False)


## Rangos plausibles (sanity checks)

`basic_range_checks` + percentiles de `DAYS_BIRTH`. Rango `DAYS_BIRTH` ampliado a (-30000, 0).


In [ ]:
range_hints = {
    "EXT_SOURCE_1": (0.0, 1.0),
    "EXT_SOURCE_2": (0.0, 1.0),
    "EXT_SOURCE_3": (0.0, 1.0),
    "DAYS_BIRTH": (-30000, 0),
    "DAYS_EMPLOYED": (-25000, 365243),
}

present = {k: v for k, v in range_hints.items() if k in train.columns}
chk = basic_range_checks(train, present)
for col, ok in sorted(chk.items()):
    print(f"{col}: {'OK' if ok else 'REVISAR'}")

if "DAYS_BIRTH" in train.columns:
    db = train["DAYS_BIRTH"].dropna()
    print("DAYS_BIRTH min/max/p01/p99:", float(db.min()), float(db.max()), float(db.quantile(0.01)), float(db.quantile(0.99)))

if "DAYS_EMPLOYED" in train.columns:
    special = (train["DAYS_EMPLOYED"] == 365243).mean()
    print(f"Fracción DAYS_EMPLOYED == 365243 (placeholder): {special:.4f}")


## Asociación con `TARGET`

Pearson en numéricas; heatmap entre top correlaciones; **Cramér V** y chi² en `object`; tasas de default por grupo.


In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.tolist()
if y_col in num_cols:
    num_cols.remove(y_col)

corr = train[num_cols + [y_col]].corr(numeric_only=True)[y_col].drop(labels=[y_col], errors="ignore")
corr = corr.replace([np.inf, -np.inf], np.nan).dropna()
corr_sorted = corr.reindex(corr.abs().sort_values(ascending=False).index)

print("Top 15 correlación con TARGET (Pearson):")
display(corr_sorted.head(15).to_frame("corr_with_TARGET"))
print("Bottom 5 (menor |corr|):")
display(corr_sorted.tail(5).to_frame("corr_with_TARGET"))

top_k = 25
top_for_heatmap = [c for c in corr_sorted.head(top_k).index if c in train.columns]
sub = train[top_for_heatmap + [y_col]].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sub.values, aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(sub.columns)))
ax.set_yticks(range(len(sub.columns)))
ax.set_xticklabels(sub.columns, rotation=90, fontsize=7)
ax.set_yticklabels(sub.columns, fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_title(f"Correlaciones: top {top_k} por |corr| con TARGET + TARGET")
plt.tight_layout()


def cramers_v(conf: np.ndarray) -> float:
    chi2, _, _, _ = chi2_contingency(conf, correction=False)
    n = conf.sum()
    r, k = conf.shape
    return float(np.sqrt(chi2 / (n * (min(r, k) - 1))))


cat_rows = []
for col in obj_cols:
    tab = pd.crosstab(train[col].fillna("__NA__"), train[y_col])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        continue
    try:
        chi2, p, _, _ = chi2_contingency(tab.values)
        cv = cramers_v(tab.values.astype(float))
        cat_rows.append({"column": col, "chi2_pvalue": p, "cramers_v": cv})
    except ValueError:
        continue
cat_assoc = pd.DataFrame(cat_rows).sort_values("cramers_v", ascending=False)
print("Asociación categórica vs TARGET (Cramér V):")
display(cat_assoc)

for col in ["CODE_GENDER", "NAME_INCOME_TYPE", "FLAG_OWN_CAR"]:
    if col not in train.columns:
        continue
    t = train.groupby(col, dropna=False)[y_col].agg(["mean", "count"]).rename(columns={"mean": "default_rate", "count": "n"})
    t = t.sort_values("n", ascending=False)
    print("Tasa de default por", col)
    display(t.head(15))


### Distribuciones `EXT_SOURCE_*` por `TARGET`


In [ ]:
ext_cols = [c for c in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"] if c in train.columns]
if not ext_cols:
    print("No hay EXT_SOURCE en train.")
else:
    fig, axes = plt.subplots(1, len(ext_cols), figsize=(4 * len(ext_cols), 3.5))
    if len(ext_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, ext_cols):
        for tval, label, color in [(0, "TARGET=0", "#4c78a8"), (1, "TARGET=1", "#f58518")]:
            s = train.loc[train[y_col] == tval, col].dropna()
            s = s[(s >= 0) & (s <= 1)]
            ax.hist(s, bins=40, alpha=0.55, label=label, color=color, density=True)
        ax.set_title(col)
        ax.set_xlabel("valor")
        ax.legend(fontsize=8)
    plt.suptitle("EXT_SOURCE (recorte [0,1])", y=1.02)
    plt.tight_layout()


### Tabla `bureau`: filas por cliente

Registros en `bureau.csv` por `SK_ID_CURR` y default por decil (aprox.).


In [ ]:
bureau_ids = read_table("bureau", usecols=[k_curr])
n_per = bureau_ids.groupby(k_curr).size().rename("n_bureau_records")
train_nb = train[[k_curr, y_col]].merge(n_per, on=k_curr, how="left")
train_nb["n_bureau_records"] = train_nb["n_bureau_records"].fillna(0)

print(train_nb["n_bureau_records"].describe(percentiles=[0.5, 0.9, 0.99]))
print("Tasa de default por decil de n_bureau_records (aprox.):")
train_nb["decile"] = pd.qcut(train_nb["n_bureau_records"], q=10, duplicates="drop")
display(train_nb.groupby("decile", observed=True)[y_col].agg(["mean", "count"]))


## Auditoría preliminar de leakage (`application_*`)

Revisar nombres y agregaciones temporales en tablas auxiliares en el notebook 02.


In [ ]:
sus = [
    c
    for c in train.columns
    if any(x in c.upper() for x in ("TARGET", "DEFAULT", "FUTURE", "POST"))
]
print("Columnas a revisar por nombre:", [c for c in sus if c != y_col])

doc_flags = [c for c in train.columns if "FLAG_DOCUMENT" in c]
print("FLAG_DOCUMENT_*:", len(doc_flags), "columnas")


## Reproducibilidad

Valores de `configs/base.yaml`.


In [ ]:
base_path = project_root() / "configs" / "base.yaml"
with open(base_path, encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

print("seed:", base_cfg.get("seed"))
print("splits:", base_cfg.get("splits"))
print("métricas:", base_cfg.get("metrics"))


## Resumen y siguientes pasos

| Hallazgo | Implicación |
|----------|-------------|
| Clase minoritaria | Estratificar CV; PR-AUC y calibración |
| Nulos por bloques | Imputación / NaN-friendly |
| NA distinto entre TARGET 0/1 | Missing como señal |
| KS train vs test alto | Deriva de datos |
| Tablas auxiliares largas | Agregaciones por cliente + tiempo |
| EXT_SOURCE / DAYS_* con corr | Features fuertes |
| Cramér V alto | Encoding / árboles |

Siguiente: `02_feature_engineering.ipynb`.
